# 03 Attention Maps: Understanding Cross-Attention

Visualize attention weights from DynaMo's cross-attention layer:
- Which residues attend to each other?
- Do binding residues cluster in representation space?
- How do different heads specialize?

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, '/home/claude/pmp_research')
from src.evaluation.interpretability import AttentionMapVisualizer, RepresentationVisualizer

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 10)

## 1. Load Model and Extract Attention

In [ ]:
# Placeholder: Load trained DynaMo model
# model = load_checkpoint('outputs/checkpoints/dynamo_best_mcc_0.7234.pt')
# model.eval()

print('Model loaded (placeholder)')

# Generate synthetic attention for demo
# In practice: visualizer.extract_attention(H_geom, H_star)
N = 250  # protein length
n_heads = 8

# Create example attention: higher along diagonals + some long-range
attn_weights = torch.zeros(n_heads, N, N)
for h in range(n_heads):
    # Diagonal bias (local attention)
    for i in range(N):
        for j in range(max(0, i-5), min(N, i+6)):
            attn_weights[h, i, j] += torch.randn(1).abs() * 0.3
    
    # Some long-range attention
    for _ in range(20):
        i, j = torch.randint(0, N, (2,)).tolist()
        attn_weights[h, i, j] += torch.randn(1).abs() * 0.5
    
    # Normalize
    attn_weights[h] = torch.softmax(attn_weights[h], dim=1)

print(f'Attention shape: {attn_weights.shape}')  # (8, 250, 250)
print(f'Mean attention value: {attn_weights.mean():.4f}')

## 2. Single Head Heatmap

In [ ]:
# Visualize head 0
fig, ax = plt.subplots(figsize=(12, 10))

attn_head = attn_weights[0].numpy()

im = ax.imshow(attn_head, cmap='Blues', aspect='auto', interpolation='nearest')

ax.set_xlabel('Key Residue Index', fontsize=12, fontweight='bold')
ax.set_ylabel('Query Residue Index', fontsize=12, fontweight='bold')
ax.set_title('Cross-Attention Weights: Head 0', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Attention Weight', fontsize=11, fontweight='bold')

# Add grid for readability
ax.set_xticks(np.arange(0, N, 25))
ax.set_yticks(np.arange(0, N, 25))
ax.grid(True, alpha=0.1, color='white', linewidth=0.5)

plt.tight_layout()
plt.savefig('outputs/attention_head_0_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ Head 0 heatmap saved')

## 3. All Heads Comparison Grid

In [ ]:
# Plot all 8 heads for comparison
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for head_idx in range(n_heads):
    attn_head = attn_weights[head_idx].numpy()
    ax = axes[head_idx]
    
    im = ax.imshow(attn_head, cmap='Blues', aspect='auto', interpolation='nearest')
    ax.set_title(f'Head {head_idx}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Key', fontsize=10)
    ax.set_ylabel('Query', fontsize=10)
    
    # Reduce ticks for clarity
    ax.set_xticks(np.arange(0, N, 50))
    ax.set_yticks(np.arange(0, N, 50))
    
    # Compute statistics for this head
    local_focus = attn_head.diagonal().mean()
    ax.text(0.98, 0.02, f'Local: {local_focus:.3f}',
           transform=ax.transAxes, ha='right', va='bottom',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.7),
           fontsize=8, fontweight='bold')

plt.suptitle('Cross-Attention: All 8 Heads', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('outputs/attention_all_heads_grid.png', dpi=300, bbox_inches='tight')
plt.show()
print('✓ All heads grid saved')

## 4. Attention Pattern Analysis

In [ ]:
# Analyze attention patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Average attention per head
mean_attn_per_head = attn_weights.mean(dim=(1, 2)).numpy()
axes[0, 0].bar(range(n_heads), mean_attn_per_head, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Head Index', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Mean Attention Weight', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Average Attention per Head', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Diagonal focus (self-attention)
diagonal_focus = []
for h in range(n_heads):
    diag = torch.diagonal(attn_weights[h])
    diagonal_focus.append(diag.mean().item())

axes[0, 1].bar(range(n_heads), diagonal_focus, color='coral', edgecolor='black')
axes[0, 1].set_xlabel('Head Index', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Diagonal Attention Weight', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Self-Attention Focus per Head', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')
axes[0, 1].axhline(y=np.mean(diagonal_focus), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 1].legend()

# 3. Local vs Long-range attention
local_range = 5  # within 5 residues
local_attn = []
long_range_attn = []

for h in range(n_heads):
    local_sum = 0
    long_range_sum = 0
    count_local = 0
    count_long = 0
    
    for i in range(N):
        for j in range(N):
            dist = abs(i - j)
            if dist <= local_range:
                local_sum += attn_weights[h, i, j].item()
                count_local += 1
            else:
                long_range_sum += attn_weights[h, i, j].item()
                count_long += 1
    
    local_attn.append(local_sum / count_local if count_local > 0 else 0)
    long_range_attn.append(long_range_sum / count_long if count_long > 0 else 0)

x = np.arange(n_heads)
width = 0.35
axes[1, 0].bar(x - width/2, local_attn, width, label='Local (±5)', color='lightgreen', edgecolor='black')
axes[1, 0].bar(x + width/2, long_range_attn, width, label='Long-range', color='lightcoral', edgecolor='black')
axes[1, 0].set_xlabel('Head Index', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Average Attention Weight', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Local vs Long-Range Attention', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].set_xticks(x)

# 4. Attention entropy (specialization)
entropy = []
for h in range(n_heads):
    attn_flat = attn_weights[h].flatten()
    # Entropy: -sum(p * log(p))
    ent = -(attn_flat * torch.log(attn_flat + 1e-8)).sum().item()
    entropy.append(ent)

axes[1, 1].bar(range(n_heads), entropy, color='mediumpurple', edgecolor='black')
axes[1, 1].set_xlabel('Head Index', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Entropy (bits)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Attention Entropy (Lower = More Specialized)', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('outputs/attention_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Attention analysis saved')

## 5. Binding-Aware Attention Analysis

In [ ]:
# Simulate binding labels
binding_residues = np.concatenate([
    np.ones(20),   # region 1
    np.zeros(50),  # gap
    np.ones(15),   # region 2
    np.zeros(165)  # rest
]).astype(bool)

non_binding = ~binding_residues

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Aggregate attention across all heads
attn_avg = attn_weights.mean(dim=0).numpy()

# 1. Binding-to-binding attention
if binding_residues.sum() > 0:
    binding_idx = np.where(binding_residues)[0]
    binding_to_binding = attn_avg[np.ix_(binding_idx, binding_idx)]
    
    axes[0].imshow(binding_to_binding, cmap='Reds', aspect='auto')
    axes[0].set_xlabel('Binding Residue Index', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Binding Residue Index', fontsize=11, fontweight='bold')
    axes[0].set_title('Binding-to-Binding Attention', fontsize=12, fontweight='bold')
    
    print(f'Binding-to-binding attention: {binding_to_binding.mean():.4f}')

# 2. Binding residue attention pattern
if binding_residues.sum() > 0 and non_binding.sum() > 0:
    binding_idx = np.where(binding_residues)[0]
    non_binding_idx = np.where(non_binding)[0]
    
    binding_to_non = attn_avg[np.ix_(binding_idx, non_binding_idx)].mean(axis=0)
    
    # Plot attention from all residues to binding residues
    attn_to_binding = attn_avg[:, binding_idx].mean(axis=1)
    
    axes[1].plot(attn_to_binding, linewidth=2, color='darkred', label='Attention to binding residues')
    
    # Highlight binding regions
    for i in range(N):
        if binding_residues[i]:
            axes[1].axvspan(i-0.5, i+0.5, alpha=0.1, color='red')
    
    axes[1].set_xlabel('Residue Index', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Average Attention Weight', fontsize=11, fontweight='bold')
    axes[1].set_title('How Much Each Residue Attends to Binding Residues', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('outputs/attention_binding_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Binding-aware attention analysis saved')

## 6. Summary Statistics

In [ ]:
print('\n' + '='*60)
print('ATTENTION ANALYSIS SUMMARY')
print('='*60)

print(f'\nModel: DynaMo with {n_heads} attention heads')
print(f'Protein length: {N} residues')
print(f'Binding residues: {binding_residues.sum()} ({binding_residues.mean()*100:.1f}%)')

print(f'\nAttention patterns:')
print(f'  Mean attention weight: {attn_weights.mean():.4f}')
print(f'  Max attention weight: {attn_weights.max():.4f}')
print(f'  Std attention weight: {attn_weights.std():.4f}')

print(f'\nHead specialization:')
for h in range(n_heads):
    local = local_attn[h]
    long = long_range_attn[h]
    pattern = 'LOCAL' if local > long else 'LONG-RANGE'
    print(f'  Head {h}: {pattern:12s} (Local={local:.4f}, Long-range={long:.4f})')

print(f'\nBinding attention:')
if binding_residues.sum() > 0:
    binding_idx = np.where(binding_residues)[0]
    binding_attn = attn_avg[np.ix_(binding_idx, binding_idx)].mean()
    non_binding_attn = attn_avg[np.ix_(~binding_residues, ~binding_residues)].mean() if non_binding.sum() > 1 else 0
    
    print(f'  Binding-to-binding: {binding_attn:.4f}')
    print(f'  Non-binding-to-non-binding: {non_binding_attn:.4f}')
    print(f'  Ratio: {binding_attn/max(non_binding_attn, 1e-6):.2f}x')

print('\n' + '='*60)